# Tutorial 01: HLLSet — Core HyperLogLog Set Operations

This tutorial introduces the `HLLSet` module, the foundational data structure for probabilistic set operations in the HLLSet Algebra library.

## What You'll Learn

1. **Creating HLLSets** — From batches of tokens
2. **Cardinality Estimation** — Approximate unique element count
3. **Set Operations** — Union, intersection, difference, XOR
4. **Hash Configuration** — Understanding the hashing system
5. **Content Addressing** — SHA1-based naming

## Key Concepts

- **HLLSet** is **immutable** — all operations return new instances
- Uses a **C/Cython backend** for performance
- Implements **MurmurHash64A** for consistent hashing
- Each HLLSet has `2^p_bits` registers (default: 1024 with p_bits=10)

In [1]:
# Import the HLLSet module
import sys
sys.path.insert(0, '..')

from core.hllset import (
    HLLSet, 
    HashConfig, 
    HashType, 
    DEFAULT_HASH_CONFIG,
    compute_sha1
)
import numpy as np

## 1. Creating HLLSets

The primary way to create an HLLSet is using `from_batch()`. This processes all tokens in a single batch operation.

In [2]:
# Create an HLLSet from a list of tokens
tokens = ["hello", "world", "foo", "bar", "hello"]  # Note: "hello" appears twice
hll = HLLSet.from_batch(tokens)

print(f"Created HLLSet with p_bits={hll.p_bits}")
print(f"Number of registers: {2**hll.p_bits}")
print(f"Content-addressed name: {hll.name[:16]}...")

Created HLLSet with p_bits=10
Number of registers: 1024
Content-addressed name: 6df7dfce04f2e306...


In [3]:
# Create HLLSets from different data sources
from_list = HLLSet.from_batch(["a", "b", "c"])
from_set = HLLSet.from_batch({"x", "y", "z"})  # Sets work too
from_generator = HLLSet.from_batch(f"token_{i}" for i in range(100))

print(f"From list: {from_list.cardinality():.1f} elements")
print(f"From set: {from_set.cardinality():.1f} elements")
print(f"From generator: {from_generator.cardinality():.1f} elements")

From list: 4.0 elements
From set: 4.0 elements
From generator: 106.0 elements


## 2. Cardinality Estimation

HLLSets provide **probabilistic cardinality estimation** with ~1-2% standard error.

The `cardinality()` method uses the HyperLogLog algorithm to estimate the number of unique elements.

In [4]:
# Demonstrate cardinality estimation
# Create HLLSet with known number of unique tokens
num_unique = 10000
tokens = [f"token_{i}" for i in range(num_unique)]
hll = HLLSet.from_batch(tokens)

estimated = hll.cardinality()
error_pct = abs(estimated - num_unique) / num_unique * 100

print(f"Actual unique tokens: {num_unique}")
print(f"Estimated cardinality: {estimated:.0f}")
print(f"Error: {error_pct:.2f}%")

Actual unique tokens: 10000
Estimated cardinality: 10211
Error: 2.11%


In [ ]:
# Duplicates don't affect cardinality
with_duplicates = ["hello"] * 1000 + ["world"] * 500 + ["foo"] * 250
hll_dupes = HLLSet.from_batch(with_duplicates)

print(f"Total tokens (with duplicates): {len(with_duplicates)}")
print(f"Estimated unique: {hll_dupes.cardinality():.1f}")

## 3. Set Operations

HLLSets support all standard set operations. Since HLLSets are **immutable**, each operation returns a **new** HLLSet.

| Operation | Method | Symbol | Description |
|-----------|--------|--------|-------------|
| Union | `union()` | ∪ | Elements in A OR B |
| Intersection | `intersect()` | ∩ | Elements in A AND B |
| Difference | `diff()` | \ | Elements in A but NOT B |
| Symmetric Diff | `xor()` | ⊕ | Elements in A XOR B |

In [ ]:
# Create two HLLSets with some overlap
set_a = HLLSet.from_batch(["apple", "banana", "cherry", "date"])
set_b = HLLSet.from_batch(["cherry", "date", "elderberry", "fig"])

print(f"Set A cardinality: {set_a.cardinality():.0f}")
print(f"Set B cardinality: {set_b.cardinality():.0f}")

In [ ]:
# Union: A ∪ B
union_ab = set_a.union(set_b)
print(f"Union (A ∪ B): {union_ab.cardinality():.0f} elements")

# Intersection: A ∩ B
intersect_ab = set_a.intersect(set_b)
print(f"Intersection (A ∩ B): {intersect_ab.cardinality():.0f} elements")

# Difference: A \ B
diff_ab = set_a.diff(set_b)
print(f"Difference (A \\ B): {diff_ab.cardinality():.0f} elements")

# Symmetric Difference: A ⊕ B (XOR)
xor_ab = set_a.xor(set_b)
print(f"Symmetric Diff (A ⊕ B): {xor_ab.cardinality():.0f} elements")

In [ ]:
# Merge multiple HLLSets at once
sets = [
    HLLSet.from_batch([f"batch1_{i}" for i in range(100)]),
    HLLSet.from_batch([f"batch2_{i}" for i in range(100)]),
    HLLSet.from_batch([f"batch3_{i}" for i in range(100)]),
]

merged = HLLSet.merge(sets)
print(f"Merged {len(sets)} HLLSets: ~{merged.cardinality():.0f} unique elements")

## 4. Hash Configuration

HLLSet is the **single source of truth** for hash configuration across the system.

The default configuration uses:
- **MurmurHash64A** for fast, high-quality hashing
- **p_bits=10** → 1024 registers
- **seed=42** for reproducibility

In [ ]:
# View the default hash configuration
config = DEFAULT_HASH_CONFIG
print(f"Hash type: {config.hash_type}")
print(f"Precision bits: {config.p_bits}")
print(f"Seed: {config.seed}")
print(f"Hash bits: {config.h_bits}")
print(f"Number of registers: {config.num_registers}")

In [ ]:
# Computing hash values
token = "hello_world"

# Class-level hash (uses default config)
h = HLLSet.hash(token)
print(f"Hash of '{token}': {h}")

# Get register index and trailing zeros
reg, zeros = HLLSet.hash_to_reg_zeros(token)
print(f"Register index: {reg}")
print(f"Trailing zeros: {zeros}")

In [ ]:
# Create HLLSet with custom configuration
custom_config = HashConfig(
    hash_type=HashType.MURMUR3,
    p_bits=12,  # 4096 registers (higher precision)
    seed=12345,
    h_bits=64
)

hll_custom = HLLSet.from_batch(["test", "tokens"], config=custom_config)
print(f"Custom HLLSet p_bits: {hll_custom.p_bits}")
print(f"Custom HLLSet config: {hll_custom.config}")

## 5. Content Addressing

Each HLLSet is automatically named using the **SHA1 hash** of its register state.

This provides:
- **Deterministic naming** — same content → same name
- **Collision resistance** — different content → different names
- **Efficient comparison** — compare names instead of full register arrays

In [ ]:
# Content-addressed naming
hll1 = HLLSet.from_batch(["a", "b", "c"])
hll2 = HLLSet.from_batch(["a", "b", "c"])  # Same content
hll3 = HLLSet.from_batch(["x", "y", "z"])  # Different content

print(f"HLL1 name: {hll1.name}")
print(f"HLL2 name: {hll2.name}")
print(f"HLL3 name: {hll3.name}")
print(f"\nHLL1 == HLL2 (same content): {hll1.name == hll2.name}")
print(f"HLL1 == HLL3 (diff content): {hll1.name == hll3.name}")

In [ ]:
# Access raw register data
hll = HLLSet.from_batch(["hello", "world"])

# Get registers as numpy array
registers = hll.dump_numpy()
print(f"Register array shape: {registers.shape}")
print(f"Register dtype: {registers.dtype}")
print(f"Non-zero registers: {np.count_nonzero(registers)}")

# Popcount (total set bits)
popcount = sum(int(r).bit_count() for r in registers)
print(f"Total set bits (popcount): {popcount}")

## 6. Batch and Parallel Processing

HLLSets support efficient batch processing and can merge results from parallel operations.

In [ ]:
# Processing multiple batches
batch1 = [f"doc1_token_{i}" for i in range(1000)]
batch2 = [f"doc2_token_{i}" for i in range(1000)]
batch3 = [f"doc3_token_{i}" for i in range(1000)]

# Process each batch independently (can be parallelized)
hll1 = HLLSet.from_batch(batch1)
hll2 = HLLSet.from_batch(batch2)
hll3 = HLLSet.from_batch(batch3)

# Merge results
combined = HLLSet.merge([hll1, hll2, hll3])
print(f"Combined cardinality: ~{combined.cardinality():.0f}")

## Summary

In this tutorial, you learned:

1. **HLLSet Creation** — Use `HLLSet.from_batch()` for immutable HLLSets
2. **Cardinality** — Probabilistic counting with ~1-2% error
3. **Set Operations** — Union, intersection, difference, XOR
4. **Hash Config** — MurmurHash64A with configurable precision
5. **Content Addressing** — SHA1-based deterministic naming

### Next Steps

- **Tutorial 02**: HLLTensor — 2D tensor view for disambiguation
- **Tutorial 03**: HLLLattice — Temporal lattice structures
- **Tutorial 04**: HLLSetStore — Persistent storage with derivation tracking